# ARE-005.1 - Leakage-safe ML Dataset Preparation
Local ARE-004 Parquet -> user-disjoint split -> raw context/target -> positives -> train-only
filtering -> train/warm-context CSR artifacts. No ALS fitting or API calls.

Install `pip install -e ".[dev,data,research,ml]"`; select the local .venv kernel.
Run all cells in order. All algorithms and validation live in src/anime_recommender.
Generated files stay in ignored data/processed/ml/. Clear outputs before saving
this notebook; do not commit raw interaction tables in outputs.


In [ ]:
import json
import pandas as pd
from IPython.display import display
from anime_recommender.exploration import project_root
from anime_recommender.ingestion.processed_data import load_processed_data
from anime_recommender.preprocessing.implicit_feedback import positive_feedback
from anime_recommender.evaluation.training_data import training_statistics
from anime_recommender.training.dataset import run_pipeline, load_dataset

root = project_root()
config_path = root / "configs/ml_dataset.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
rebuild = True


## Original ratings and positive thresholds
Source ratings are not overwritten. Missing and zero ratings are not positives.
Default threshold is 5; thresholds 5, 6, 7 and 8 are supported; uniform confidence is 1 in this version.


In [ ]:
source = load_processed_data(root / config["input_dir"])
source_stats = training_statistics(source.interactions)
display(source_stats["ratings"])
prepared = run_pipeline(config, root) if rebuild else load_dataset(root / config["output_dir"], expected_config=config, expected_source_sha256=source.quality_report["output_sha256"])
display(prepared.tables["threshold_comparison"])
positives = positive_feedback(source.interactions, config["positive_threshold"])
positive_stats = training_statistics(positives)
display(positive_stats["summary"])
display(positive_stats["per_user"].describe())
display(positive_stats["per_anime"].describe())


## User cohorts and train-only filtering
All source users are allocated 80/10/10 by seeded hash order BEFORE activity filters.
Only TRAIN users require >=5 positives. Train is filtered to a fixed point with >=5 users/item
and >=5 items/user; only train decides popularity and vocabulary. Cohorts are not
rebalanced afterward. Filtered-out train users remain recorded in user_assignments.


In [ ]:
display(pd.DataFrame(prepared.report["cohorts"]).T)
display(pd.Series(prepared.report["train"]))
display(pd.DataFrame(prepared.report["train"]["filter_rounds"]))
train_stats = training_statistics(prepared.tables["train_interactions"])
display(train_stats["per_user"].describe())
display(train_stats["per_anime"].describe())
display(prepared.tables["train_interactions"].head())


## Fold-in context, hidden targets and cold items
All assigned eval users are kept. Raw cleaned histories are split approximately
80/20 BEFORE applying rating>=5 separately to context and target. No positive-count
activity filter is applied to eval. Empty positive targets/context are allowed and
reported. Both positive tables retain OOV items with is_cold_item=True.
Future fold-in MUST use only non-cold context, never target. Users with no known
context remain accounted for, not silently dropped. Saved context CSR matrices contain ONLY warm positive context; all cohort users
get rows, including zero rows. Unknown targets are unavoidable
misses for an ALS-only model and must remain visible in end-to-end evaluation.


In [ ]:
display(pd.DataFrame(prepared.report["evaluation"]).T)
display(pd.DataFrame({"context": prepared.report["evaluation"]["validation"]["context"], "target": prepared.report["evaluation"]["validation"]["target"]}).T)
display(pd.DataFrame({"context": prepared.report["evaluation"]["test"]["context"], "target": prepared.report["evaluation"]["test"]["target"]}).T)
display(prepared.tables["validation_context"].head())
display(prepared.tables["validation_target"].head())

display(prepared.tables["validation_user_mapping"].head())
display(prepared.tables["test_user_mapping"].head())


## Sparse matrix and reusable mappings
Matrix rows are training users, columns are dataset anime IDs. The item mapping
also carries MAL ID for future online inference. Never use raw MAL IDs as columns.
No dense user-item matrix is constructed. Binary positive confidence is 1.


In [ ]:
display(pd.Series(prepared.report["matrix"]))
display(prepared.tables["train_user_mapping"].head())
display(prepared.tables["item_mapping"].head())
display(pd.Series(prepared.report["checks"], name="Passed"))
display(prepared.report["warnings"])

display(prepared.report["evaluation"]["validation"]["context_csr"])
display(prepared.report["evaluation"]["test"]["context_csr"])


## Limitations
No interaction timestamps: this is random preference reconstruction, not temporal
prediction. The cohort allocation is stable for fixed input users/seed; changing
that population can change assignments. Other thresholds change positives, but not raw eval partition membership.
One thousand source users are a development sample, not an audience-wide benchmark.
OOV shares are reported for both unique items and interactions. Conditional metrics
on known items alone must not be presented as full-service quality.

No recommendations, model factors, ranking, split-dependent popularity from eval,
MLflow or external services are implemented. Save with all outputs cleared.
